In [2]:
from __future__ import annotations

import subprocess
from datetime import datetime, timedelta
from pathlib import Path

import pandas as pd

from fraud_stream.config import RAW_REPO_DIR, RAW_DATA_DIR, RAW_REPO_URL


def ensure_raw_repo() -> Path:
    if RAW_REPO_DIR.exists():
        return RAW_DATA_DIR

    RAW_REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
    cmd = ["git", "clone", "--depth", "1", RAW_REPO_URL, str(RAW_REPO_DIR)]
    subprocess.run(cmd, check=True)


def _date_range(begin_date: str, end_date: str):
    begin = datetime.strptime(begin_date, "%Y-%m-%d").date()
    end = datetime.strptime(end_date, "%Y-%m-%d").date()
    current = begin
    while current <= end:
        yield current.strftime("%Y-%m-%d")
        current += timedelta(days=1)


def read_from_daily_pickles(data_dir: Path, begin_date: str, end_date: str) -> pd.DataFrame:
    frames = []
    missing = []
    for day in _date_range(begin_date, end_date):
        path = data_dir / f"{day}.pkl"
        if not path.exists():
            missing.append(path.name)
            continue
        frames.append(pd.read_pickle(path))

    if not frames:
        raise FileNotFoundError(
            f"No .pkl files found in {data_dir} for period {begin_date}..{end_date}. "
            "Run ensure_raw_repo() or check dates."
        )

    if missing:
        print(f"[WARN] Missing {len(missing)} daily files. First missing: {missing[:5]}")

    df = pd.concat(frames, ignore_index=True)
    df["TX_DATETIME"] = pd.to_datetime(df["TX_DATETIME"])
    df = df.sort_values("TX_DATETIME").reset_index(drop=True)

    # Some files already contain TRANSACTION_ID, but make it monotonic after concat.
    if "TRANSACTION_ID" not in df.columns:
        df.insert(0, "TRANSACTION_ID", range(len(df)))
    return df


def load_transactions(begin_date: str = "2018-04-01", end_date: str = "2018-07-09") -> pd.DataFrame:
    data_dir = ensure_raw_repo()
    return read_from_daily_pickles(data_dir, begin_date, end_date)


if __name__ == "__main__":
    df = load_transactions()
    print(df.head())
    print(df.shape)
    print(df["TX_FRAUD"].value_counts(dropna=False))


ImportError: Unable to import required dependency numpy. Please see the traceback for details.

In [2]:
df

,TRANSACTION_ID,TX_DATETIME,CUSTOMER_ID,TERMINAL_ID,TX_AMOUNT,TX_TIME_SECONDS,TX_TIME_DAYS,TX_FRAUD,TX_FRAUD_SCENARIO
0,0,2018-04-01 00:00:31,596,3156,57.16,31,0,0,0
1,1,2018-04-01 00:02:10,4961,3412,81.51,130,0,0,0
2,2,2018-04-01 00:07:56,2,1365,146.00,476,0,0,0
3,3,2018-04-01 00:09:29,4128,8737,64.49,569,0,0,0
4,4,2018-04-01 00:10:34,927,9906,50.99,634,0,0,0
...,...,...,...,...,...,...,...,...,...
959224,959224,2018-07-09 23:53:37,3698,6046,40.14,8639617,99,0,0
959225,959225,2018-07-09 23:53:39,3565,9889,13.42,8639619,99,0,0
959226,959226,2018-07-09 23:57:39,3373,5963,67.22,8639859,99,0,0
959227,959227,2018-07-09 23:58:24,187,9772,47.29,8639904,99,0,0


In [3]:
from fraud_stream.features.build_features import build_feature_table
import numpy as np
import pandas as pd


def generate_customer_profiles_table(n_customers, random_state=0):
    np.random.seed(random_state)
    rows = []
    for customer_id in range(n_customers):
        x_customer_id = np.random.uniform(0, 100)
        y_customer_id = np.random.uniform(0, 100)
        mean_amount = np.random.uniform(5, 100)
        std_amount = mean_amount / 2
        mean_nb_tx_per_day = np.random.uniform(0, 4)
        rows.append([
            customer_id,
            x_customer_id,
            y_customer_id,
            mean_amount,
            std_amount,
            mean_nb_tx_per_day
        ])
    return pd.DataFrame(rows, columns=[
        "CUSTOMER_ID",
        "x_customer_id",
        "y_customer_id",
        "mean_amount",
        "std_amount",
        "mean_nb_tx_per_day"
    ])


def generate_terminal_profiles_table(n_terminals, random_state=1):
    np.random.seed(random_state)
    rows = []
    for terminal_id in range(n_terminals):
        x_terminal_id = np.random.uniform(0, 100)
        y_terminal_id = np.random.uniform(0, 100)
        rows.append([terminal_id, x_terminal_id, y_terminal_id])
    return pd.DataFrame(rows, columns=[
        "TERMINAL_ID",
        "x_terminal_id",
        "y_terminal_id"
    ])


customers = generate_customer_profiles_table(
    n_customers=5000,
    random_state=0
)

terminals = generate_terminal_profiles_table(
    n_terminals=10000,
    random_state=1
)

df_geo = df.merge(
    customers[["CUSTOMER_ID", "x_customer_id", "y_customer_id"]],
    on="CUSTOMER_ID",
    how="left"
)

df_geo = df_geo.merge(
    terminals[["TERMINAL_ID", "x_terminal_id", "y_terminal_id"]],
    on="TERMINAL_ID",
    how="left"
)

df_geo["distance_customer_terminal"] = np.sqrt(
    (df_geo["x_customer_id"] - df_geo["x_terminal_id"]) ** 2
    + (df_geo["y_customer_id"] - df_geo["y_terminal_id"]) ** 2
)

df_geo = build_feature_table(df_geo)

In [8]:
df_geo = df_geo.drop(columns=[
    "x_customer_id",
    "y_customer_id",
    "x_terminal_id",
    "y_terminal_id"
])

In [9]:
(df_geo["distance_customer_terminal"] <= 5).mean()

np.float64(1.0)

In [17]:
df_geo.describe()

,TRANSACTION_ID,TX_DATETIME,TX_AMOUNT,TX_FRAUD,TX_FRAUD_SCENARIO,distance_customer_terminal,hour,dayofweek,is_weekend,is_night,...,time_since_last_customer_tx,terminal_tx_count_1h,customer_terminal_seen_before,customer_terminal_tx_count_30d,terminal_nb_tx_1d_delay7,terminal_risk_1d_delay7,terminal_nb_tx_7d_delay7,terminal_risk_7d_delay7,terminal_nb_tx_30d_delay7,terminal_risk_30d_delay7
count,959229.00000,959229,959229.000000,959229.000000,959229.000000,959229.000000,959229.000000,959229.000000,959229.000000,959229.000000,...,9.592290e+05,959229.000000,959229.000000,959229.000000,959229.000000,959229.000000,959229.000000,959229.000000,959229.000000,959229.000000
mean,479614.00000,2018-05-20 23:52:15.191023616,53.614197,0.007940,0.017865,3.309219,11.498785,2.999373,0.290155,0.182567,...,4.365171e+04,0.053819,0.686328,0.899489,0.996253,0.004929,6.762099,0.007636,25.527210,0.006991
min,0.00000,2018-04-01 00:00:31,0.000000,0.000000,0.000000,0.011552,0.000000,0.000000,0.000000,0.000000,...,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,239807.00000,2018-04-25 22:45:38,21.010000,0.000000,0.000000,2.462127,8.000000,1.000000,0.000000,0.000000,...,7.868000e+03,0.000000,0.000000,0.000000,0.000000,0.000000,5.000000,0.000000,19.000000,0.000000
50%,479614.00000,2018-05-20 19:53:45,44.660000,0.000000,0.000000,3.506565,12.000000,3.000000,0.000000,0.000000,...,2.186200e+04,0.000000,1.000000,1.000000,1.000000,0.000000,7.000000,0.000000,26.000000,0.000000
75%,719421.00000,2018-06-15 01:23:53,76.940000,0.000000,0.000000,4.314410,15.000000,5.000000,1.000000,0.000000,...,5.720200e+04,0.000000,1.000000,1.000000,2.000000,0.000000,9.000000,0.000000,33.000000,0.000000
max,959228.00000,2018-07-09 23:59:24,1071.150000,1.000000,3.000000,4.999990,23.000000,6.000000,1.000000,1.000000,...,5.639579e+06,3.000000,1.000000,11.000000,11.000000,1.000000,25.000000,1.000000,75.000000,1.000000
std,276905.70502,NaN,42.064693,0.088751,0.206170,1.187835,5.054534,2.026181,0.453834,0.386312,...,7.877508e+04,0.232667,0.463985,1.071181,1.021594,0.066589,3.137020,0.069540,11.229989,0.054095


In [11]:
df_geo.columns

Index(['TRANSACTION_ID', 'TX_DATETIME', 'CUSTOMER_ID', 'TERMINAL_ID',
       'TX_AMOUNT', 'TX_TIME_SECONDS', 'TX_TIME_DAYS', 'TX_FRAUD',
       'TX_FRAUD_SCENARIO', 'distance_customer_terminal', 'hour', 'dayofweek',
       'is_weekend', 'is_night', 'customer_nb_tx_1d', 'customer_avg_amount_1d',
       'customer_sum_amount_1d', 'customer_nb_tx_7d', 'customer_avg_amount_7d',
       'customer_sum_amount_7d', 'customer_nb_tx_30d',
       'customer_avg_amount_30d', 'customer_sum_amount_30d',
       'amount_ratio_to_customer_avg_7d', 'amount_zscore_customer',
       'customer_tx_count_10min', 'customer_tx_count_1h',
       'customer_amount_sum_1h', 'customer_amount_max_24h',
       'amount_ratio_to_customer_median_30d', 'time_since_last_customer_tx',
       'terminal_tx_count_1h', 'customer_terminal_seen_before',
       'customer_terminal_tx_count_30d', 'terminal_nb_tx_1d_delay7',
       'terminal_risk_1d_delay7', 'terminal_nb_tx_7d_delay7',
       'terminal_risk_7d_delay7', 'terminal_nb_tx_

In [6]:
from __future__ import annotations

import pandas as pd

from fraud_stream.config import PROCESSED_DIR


def load_feature_table() -> pd.DataFrame:
    parquet_path = PROCESSED_DIR / "handbook_features.parquet"
    pickle_path = PROCESSED_DIR / "handbook_features.pkl"

    if parquet_path.exists():
        return pd.read_parquet(parquet_path)
    if pickle_path.exists():
        return pd.read_pickle(pickle_path)

    raise FileNotFoundError("Feature table not found. Run scripts/run_all.py first.")


featured_df = load_feature_table()

featured_df.columns

Index(['TRANSACTION_ID', 'TX_DATETIME', 'CUSTOMER_ID', 'TERMINAL_ID',
       'TX_AMOUNT', 'TX_TIME_SECONDS', 'TX_TIME_DAYS', 'TX_FRAUD',
       'TX_FRAUD_SCENARIO', 'hour', 'dayofweek', 'is_weekend', 'is_night',
       'customer_nb_tx_1d', 'customer_avg_amount_1d', 'customer_sum_amount_1d',
       'customer_nb_tx_7d', 'customer_avg_amount_7d', 'customer_sum_amount_7d',
       'customer_nb_tx_30d', 'customer_avg_amount_30d',
       'customer_sum_amount_30d', 'amount_ratio_to_customer_avg_7d',
       'amount_zscore_customer', 'customer_tx_count_10min',
       'customer_tx_count_1h', 'customer_amount_sum_1h',
       'customer_amount_max_24h', 'amount_ratio_to_customer_median_30d',
       'time_since_last_customer_tx', 'terminal_tx_count_1h',
       'customer_terminal_seen_before', 'customer_terminal_tx_count_30d',
       'terminal_nb_tx_1d_delay7', 'terminal_risk_1d_delay7',
       'terminal_nb_tx_7d_delay7', 'terminal_risk_7d_delay7',
       'terminal_nb_tx_30d_delay7', 'terminal_risk_30

In [16]:
df_geo.to_parquet(PROCESSED_DIR / "features.parquet", index=False)